# Review sensitivity-analysis value pairs

### Overview
This notebook provides an **interactive interface** for reviewing the sensitivity-analysis value pairs used in the publication on **the validation of *EOlife X***.  
The dataset originates from a **porcine model of cardiopulmonary resuscitation (CPR)**.

The **sensitivity analysis** pairs each breath cycle detected by *EOlife X* to the nearest reference breath by matching the peak of the *EOlife X* cumulative volume signal to the reference inspiratory or expiratory volume, yielding an alternative set of value pairs that is independent of the primary occlusion-based pairing.

### Usage Instructions
- Use the **dropdown menu** to select the case to be reviewed.  
- The corresponding physiological signals and annotations are displayed in synchronized subplots, with an **overview of the full duration** shown beneath.  
  - The **overview plot is clickable**: clicking a region updates the main display.  
  - The **highlighted box** in the overview indicates the **currently visible time window** in the main plots.  
- **Navigation controls:**  
  - Press **`+`** or **`-`** to zoom in and out along the time axis.  
  - Use the **arrow keys** to pan left or right.  
  - Hover the mouse over a subplot and scroll the **mouse wheel** to adjust the **vertical scaling** of that trace.  

### Notes
This interactive viewer facilitates **visual inspection and validation** of sensitivity-analysis value pairs obtained from raw experimental data.  
It serves as a supplementary tool to enhance **transparency**, **reproducibility**, and **understanding** of the analyses presented in the publication.

In [ ]:
from vitabel import Vitals
from pathlib import Path
import pandas as pd
from copy import deepcopy
import ipywidgets as widgets
from IPython.display import display

def plot_curves(case: Vitals, start: pd.Timestamp | None = None, stop: pd.Timestamp | None = None):
    case.get_label("Inspiration Begin").plotstyle.update(linestyle="dotted")
    case.get_label("Expiration Begin").plotstyle.update(linestyle="dashed")

    ghost_chan = deepcopy(case.get_channel("f"))
    ghost_chan.rename("ghost")
    ghost_chan.data[:] = 0
    ghost_chan.plotstyle.update(marker="", label="", alpha=0)
    case.add_channel(ghost_chan)

    for chan in case.get_channels("Expiratory Volume") + case.get_channels("Inspiratory Volume"):
        chan.plotstyle.update(alpha=0.4)

    vi_labels = ["Inspiration Begin", "Expiration Begin", "VTinsp_sens", "insp_gt"]
    ve_labels = ["Inspiration Begin", "Expiration Begin", "VTexp_sens", "exp_gt"]

    # Drop labels that are absent in this case
    present = set(case.get_label_names())
    vi_labels = [l for l in vi_labels if l in present]
    ve_labels = [l for l in ve_labels if l in present]

    plot = case.plot_interactive(
        channels=[["Flow Interpolated"],
                  ["Pressure Interpolated"],
                  ["vi", "Inspiratory Volume"],
                  ["ve", "Expiratory Volume"],
                  ["f", "ghost"]],
        labels=[["Inspiration", "Expiration"],
                ["Inspiration", "Expiration"],
                vi_labels,
                ve_labels,
                ["Inspiration Begin", "Expiration Begin", "Respiratory Rate", "rr_corr_gt", "f_gt"]],
        channel_overviews=[["Inspiratory Volume"]],
        start=start,
        stop=stop,
        subplots_kwargs={
            "figsize": (18, 10),
            "dpi": 100,
            "gridspec_kw": {"height_ratios": [1, 1, 1, 1, 1, 0.5]}
        },
        time_unit="s")

    fig = plot.center.figure
    axes = fig.get_axes()

    font_size = 10
    axes[0].set_ylabel(r"Flow", fontsize=font_size)
    axes[1].set_ylabel(r"Pressure", fontsize=font_size)
    axes[2].set_ylabel(r"$Vt_{\mathrm{i}}$", fontsize=font_size)
    axes[3].set_ylabel(r"$Vt_{\mathrm{e}}$", fontsize=font_size)
    axes[4].set_ylabel(r"$f$", fontsize=font_size)
    axes[5].set_ylabel("Overview")

    axes[3].set_ylim(axes[2].get_ylim())

    units = ["L/min", "cmH₂O", "mL", "mL", " min⁻\xb9"]
    for i, unit in enumerate(units):
        axes[i].text(1.02, 0.5, unit,
                     transform=axes[i].transAxes,
                     fontsize=font_size - 1,
                     color="grey",
                     ha="left", va="center", rotation=90)

    for ax in axes:
        legend = ax.get_legend()
        if legend:
            legend.remove()

    try:
        fig.tight_layout(rect=[0, 0.015, 0.97, 1])
    except Exception:
        pass
    fig.subplots_adjust(hspace=0)

    for ax in axes[:-2]:
        ax.set_xlabel(None)
        ax.set_xticklabels([])

    pos = axes[5].get_position()
    axes[5].set_position([pos.x0, pos.y0 - 0.03, pos.width, pos.height])

    fig.suptitle("")
    return plot


# Load cases that have sensitivity labels
cases = {}
widgets_map = {}
for case_path in sorted(Path("../data/").glob("*.json")):
    case_id = case_path.stem
    case = Vitals()
    case.load_data(case_path)
    if "VTinsp_sens" not in case.get_label_names() and "VTexp_sens" not in case.get_label_names():
        print(f"Skipping {case_id}: no sensitivity labels")
        continue
    cases[case_id] = case
    widgets_map[case_id] = plot_curves(case)

# Dropdown to switch between cases
dd = widgets.Dropdown(
    options=list(sorted(widgets_map.keys())),
    value=next(iter(sorted(widgets_map.keys()))),
    description="Select Case:",
    layout=widgets.Layout(width="180px")
)
holder = widgets.Box()

def on_select(change):
    if change["name"] == "value":
        holder.children = (widgets_map[change["new"]],)

dd.observe(on_select, names="value")
holder.children = (widgets_map[dd.value],)
display(dd, widgets.HTML("<br>"), holder)